In [ ]:
import numpy as np
import pandas as pd
import time
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [1]:
# ============================================================
# HARDWARE MONITORING
# ============================================================

!pip install -q psutil nvidia-ml-py


import os
import time
import threading
import numpy as np
import pandas as pd
import psutil



In [2]:
# ------------------------------------------------------------
# NVIDIA NVML
# ------------------------------------------------------------

try:
    import pynvml

    pynvml.nvmlInit()
    NVML_AVAILABLE = True

except Exception:
    NVML_AVAILABLE = False

In [3]:
# ------------------------------------------------------------
# Hardware Monitor
# ------------------------------------------------------------

class HardwareMonitor:

    def __init__(self, interval=0.2, gpu_index=0):

        self.interval = interval
        self.gpu_index = gpu_index

        self.running = False
        self.thread = None

        # CPU
        self.cpu_util_samples = []
        self.ram_samples = []

        # GPU
        self.gpu_util_samples = []
        self.gpu_mem_util_samples = []
        self.vram_samples = []
        self.gpu_power_samples = []
        self.gpu_temp_samples = []

        self.timestamps = []

        self.process = psutil.Process(os.getpid())

        self.gpu_handle = None

        if NVML_AVAILABLE:

            try:
                self.gpu_handle = (
                    pynvml.nvmlDeviceGetHandleByIndex(
                        gpu_index
                    )
                )

            except Exception:
                self.gpu_handle = None

    # --------------------------------------------------------
    # Start
    # --------------------------------------------------------
    def start(self):

        # Initialize process CPU counter
        self.process.cpu_percent(None)

        self.running = True

        self.thread = threading.Thread(
            target=self._monitor,
            daemon=True
        )

        self.thread.start()

    # --------------------------------------------------------
    # Monitoring loop
    # --------------------------------------------------------
    def _monitor(self):

        logical_cpus = psutil.cpu_count(
            logical=True
        )

        while self.running:

            timestamp = time.perf_counter()
            # =================================================
            # CPU UTILIZATION
            # =================================================
            process_cpu = (
                self.process.cpu_percent(
                    interval=None
                )
            )

            # Normalize process CPU usage to
            # percentage of total logical CPU capacity.
            if logical_cpus:
                process_cpu_normalized = (
                    process_cpu / logical_cpus
                )
            else:
                process_cpu_normalized = process_cpu
            self.cpu_util_samples.append(
                process_cpu_normalized
            )

            # =================================================
            # RAM
            # =================================================

            ram_mb = (
                self.process.memory_info().rss
                / (1024 ** 2)
            )

            self.ram_samples.append(
                ram_mb
            )

            # =================================================
            # GPU
            # =================================================

            if self.gpu_handle is not None:

                try:

                    utilization = (
                        pynvml.nvmlDeviceGetUtilizationRates(
                            self.gpu_handle
                        )
                    )

                    memory = (
                        pynvml.nvmlDeviceGetMemoryInfo(
                            self.gpu_handle
                        )
                    )

                    power = (
                        pynvml.nvmlDeviceGetPowerUsage(
                            self.gpu_handle
                        ) / 1000.0
                    )

                    temperature = (
                        pynvml.nvmlDeviceGetTemperature(
                            self.gpu_handle,
                            pynvml.NVML_TEMPERATURE_GPU
                        )
                    )

                    # GPU compute utilization
                    self.gpu_util_samples.append(
                        float(utilization.gpu)
                    )

                    # GPU memory-controller utilization
                    self.gpu_mem_util_samples.append(
                        float(utilization.memory)
                    )

                    # VRAM used
                    self.vram_samples.append(
                        memory.used / (1024 ** 2)
                    )

                    # Power in Watts
                    self.gpu_power_samples.append(
                        power
                    )

                    # Temperature
                    self.gpu_temp_samples.append(
                        float(temperature)
                    )

                except Exception:
                    pass

            self.timestamps.append(
                timestamp
            )

            time.sleep(
                self.interval
            )

    # --------------------------------------------------------
    # Stop
    # --------------------------------------------------------

    def stop(self):

        self.running = False

        if self.thread is not None:
            self.thread.join()

    # --------------------------------------------------------
    # Results
    # --------------------------------------------------------

    def get_results(self):

        result = {}

        # =====================================================
        # CPU
        # =====================================================

        result["avg_cpu_util_percent"] = (
            np.mean(
                self.cpu_util_samples
            )
            if self.cpu_util_samples
            else np.nan
        )

        result["peak_cpu_util_percent"] = (
            np.max(
                self.cpu_util_samples
            )
            if self.cpu_util_samples
            else np.nan
        )

        result["avg_ram_mb"] = (
            np.mean(
                self.ram_samples
            )
            if self.ram_samples
            else np.nan
        )

        result["peak_ram_mb"] = (
            np.max(
                self.ram_samples
            )
            if self.ram_samples
            else np.nan
        )

        # =====================================================
        # GPU
        # =====================================================

        if self.gpu_util_samples:

            result["avg_gpu_util_percent"] = np.mean(
                self.gpu_util_samples
            )

            result["peak_gpu_util_percent"] = np.max(
                self.gpu_util_samples
            )

            result["avg_gpu_memory_util_percent"] = np.mean(
                self.gpu_mem_util_samples
            )

            result["peak_gpu_memory_util_percent"] = np.max(
                self.gpu_mem_util_samples
            )

            result["avg_vram_mb"] = np.mean(
                self.vram_samples
            )

            result["peak_vram_mb"] = np.max(
                self.vram_samples
            )

            result["avg_gpu_power_w"] = np.mean(
                self.gpu_power_samples
            )

            result["peak_gpu_power_w"] = np.max(
                self.gpu_power_samples
            )

            result["avg_gpu_temperature_c"] = np.mean(
                self.gpu_temp_samples
            )

            result["peak_gpu_temperature_c"] = np.max(
                self.gpu_temp_samples
            )

            # =================================================
            # Energy
            # E = integral(P dt)
            # Trapezoidal approximation
            # =================================================

            energy_joules = 0.0

            n = min(
                len(self.gpu_power_samples),
                len(self.timestamps)
            )

            for i in range(1, n):

                dt = (
                    self.timestamps[i]
                    - self.timestamps[i - 1]
                )

                avg_power = (
                    self.gpu_power_samples[i]
                    + self.gpu_power_samples[i - 1]
                ) / 2.0

                energy_joules += (
                    avg_power * dt
                )

            result["gpu_energy_joules"] = (
                energy_joules
            )

        else:

            result["avg_gpu_util_percent"] = np.nan
            result["peak_gpu_util_percent"] = np.nan

            result["avg_gpu_memory_util_percent"] = np.nan
            result["peak_gpu_memory_util_percent"] = np.nan

            result["avg_vram_mb"] = np.nan
            result["peak_vram_mb"] = np.nan

            result["avg_gpu_power_w"] = np.nan
            result["peak_gpu_power_w"] = np.nan

            result["avg_gpu_temperature_c"] = np.nan
            result["peak_gpu_temperature_c"] = np.nan

            result["gpu_energy_joules"] = np.nan

        return result

In [4]:
# ------------------------------------------------------------
# Save 5-run results
# ------------------------------------------------------------

def save_five_run_results(
    run_results,
    filename,
    algorithm,
    device
):

    rows = []

    for result in run_results:

        row = result.copy()

        row["Algorithm"] = algorithm
        row["Device"] = device

        rows.append(row)

    runs_df = pd.DataFrame(rows)

    # --------------------------------------------------------
    # Calculate average
    # --------------------------------------------------------

    numeric_columns = [
        c for c in runs_df.columns
        if c not in ["run"]
        and pd.api.types.is_numeric_dtype(
            runs_df[c]
        )
    ]

    average_row = {
        "run": "AVERAGE",
        "Algorithm": algorithm,
        "Device": device
    }

    for column in numeric_columns:

        average_row[column] = (
            runs_df[column].mean()
        )

    # --------------------------------------------------------
    # Calculate standard deviation
    # --------------------------------------------------------

    std_row = {
        "run": "STD",
        "Algorithm": algorithm,
        "Device": device
    }

    for column in numeric_columns:

        std_row[column] = (
            runs_df[column].std()
        )

    final_df = pd.concat(
        [
            runs_df,
            pd.DataFrame([
                average_row,
                std_row
            ])
        ],
        ignore_index=True
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    final_df.to_csv(
        filename,
        index=False
    )

    print("\n" + "=" * 80)
    print(f"{algorithm} - {device}")
    print("=" * 80)

    display(
        final_df.round(3)
    )

    print(
        f"\nSaved result file:\n{filename}"
    )

    return final_df

In [5]:
from google.colab import drive

drive.mount("/content/drive")

RESULT_DIR = (
    "/content/drive/MyDrive/"
    "LLORMA_Project/hardware_results"
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)

print(
    "Results directory:",
    RESULT_DIR
)

Mounted at /content/drive
Results directory: /content/drive/MyDrive/LLORMA_Project/hardware_results


In [8]:
df = pd.read_csv('ratings.csv')
print(df.columns.tolist())
print(df.shape)
df.head()

['userId', 'movieId', 'rating', 'timestamp']
(100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [10]:
col_map = {}
for c in df.columns:
    lc = c.lower().replace('_', '')
    if 'user' in lc: col_map[c] = 'user'
    elif 'movie' in lc or 'item' in lc: col_map[c] = 'item'
    elif 'rating' in lc: col_map[c] = 'rating'
df = df.rename(columns=col_map)

df = df[['user', 'item', 'rating']].dropna()

user_ids = df['user'].astype('category').cat.codes.values
item_ids = df['item'].astype('category').cat.codes.values
ratings  = df['rating'].astype(np.float32).values

n_users = user_ids.max() + 1
n_items = item_ids.max() + 1
print(f"Users: {n_users}, Items: {n_items}, Ratings: {len(ratings)}")

RANDOM_SEED = 42 # Added this line to define RANDOM_SEED locally
np.random.seed(RANDOM_SEED)
idx = np.random.permutation(len(ratings))
split = int(0.8 * len(ratings))
train_idx, test_idx = idx[:split], idx[split:]

train_u, train_i, train_r = user_ids[train_idx], item_ids[train_idx], ratings[train_idx]
test_u,  test_i,  test_r  = user_ids[test_idx],  item_ids[test_idx],  ratings[test_idx]

global_mean = train_r.mean()
print(f"Train: {len(train_r)}, Test: {len(test_r)}, Global mean: {global_mean:.3f}")

Users: 610, Items: 9724, Ratings: 100836
Train: 80668, Test: 20168, Global mean: 3.503


In [13]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

train_mat = csr_matrix((train_r - global_mean, (train_u, train_i)), shape=(n_users, n_items))

EMBED_DIM = 5  # small rank, just used for distance/kernel, not final predictions
U_svd, S, Vt = svds(train_mat.asfptype(), k=EMBED_DIM)
U_svd, S, Vt = U_svd[:, ::-1], S[::-1], Vt[::-1, :]  # svds returns ascending order

U_embed = U_svd * np.sqrt(S)     # (n_users, EMBED_DIM)
V_embed = (Vt.T) * np.sqrt(S)    # (n_items, EMBED_DIM)
print("Global embedding shapes:", U_embed.shape, V_embed.shape)

Global embedding shapes: (610, 5) (9724, 5)


In [14]:
def epanechnikov_kernel(dist, bandwidth):
    """0.75 * (1 - (d/h)^2) for d < h, else 0"""
    ratio = dist / bandwidth
    return np.where(ratio < 1.0, 0.75 * (1.0 - ratio ** 2), 0.0)

In [15]:
def train_local_model(train_u, train_i, train_r, weights, mask,
                       n_users, n_items, rank=5, n_epochs=15,
                       lr=0.01, reg=0.05, seed=0, batch_size=512):
    rng = np.random.RandomState(seed)
    P = rng.normal(0, 0.1, (n_users, rank)).astype(np.float32)
    Q = rng.normal(0, 0.1, (n_items, rank)).astype(np.float32)

    u_sub, i_sub, r_sub, w_sub = train_u[mask], train_i[mask], train_r[mask], weights[mask]
    n_samples = len(u_sub)
    if n_samples == 0:
        return P, Q

    for epoch in range(n_epochs):
        perm = rng.permutation(n_samples)
        u_sub, i_sub, r_sub, w_sub = u_sub[perm], i_sub[perm], r_sub[perm], w_sub[perm]

        for start in range(0, n_samples, batch_size):
            end = start + batch_size
            bu, bi, br, bw = u_sub[start:end], i_sub[start:end], r_sub[start:end], w_sub[start:end]

            pu, qi = P[bu], Q[bi]
            pred = np.sum(pu * qi, axis=1)
            err = (br - pred) * bw

            grad_p = -2 * err[:, None] * qi + 2 * reg * pu
            grad_q = -2 * err[:, None] * pu + 2 * reg * qi

            np.add.at(P, bu, -lr * grad_p)
            np.add.at(Q, bi, -lr * grad_q)

    return P, Q

In [16]:
def predict_llorma(local_models, users, items, global_mean):
    preds = np.zeros(len(users), dtype=np.float64)
    weight_sums = np.zeros(len(users), dtype=np.float64)

    for m in local_models:
        w = m['wu'][users] * m['wi'][items]
        active = w > 1e-6
        if not np.any(active):
            continue
        local_pred = np.sum(m['P'][users[active]] * m['Q'][items[active]], axis=1)
        preds[active] += w[active] * local_pred
        weight_sums[active] += w[active]

    covered = weight_sums > 1e-6
    preds[covered] /= weight_sums[covered]
    preds[~covered] = global_mean

    return np.clip(preds, 0.5, 5.0)

In [17]:
def train_llorma(train_u, train_i, train_r, U_embed, V_embed,
                  n_users, n_items, n_anchors=20, rank=5,
                  bandwidth=0.8, n_epochs=15, lr=0.01, reg=0.05):
    rng = np.random.RandomState(RANDOM_SEED)
    anchor_positions = rng.choice(len(train_u), size=n_anchors, replace=False)
    local_models = []

    for t, pos in enumerate(anchor_positions):
        au, ai = train_u[pos], train_i[pos]

        du = np.linalg.norm(U_embed - U_embed[au], axis=1)
        di = np.linalg.norm(V_embed - V_embed[ai], axis=1)

        wu = epanechnikov_kernel(du, bandwidth)
        wi = epanechnikov_kernel(di, bandwidth)

        sample_w = wu[train_u] * wi[train_i]
        mask = sample_w > 1e-6
        if mask.sum() < 20:
            continue

        P, Q = train_local_model(train_u, train_i, train_r, sample_w, mask,
                                  n_users, n_items, rank=rank, n_epochs=n_epochs,
                                  lr=lr, reg=reg, seed=t)

        local_models.append({'au': au, 'ai': ai, 'P': P, 'Q': Q, 'wu': wu, 'wi': wi})
        print(f"Anchor {t+1}/{n_anchors} trained | neighbors used: {mask.sum()}")

    return local_models

In [18]:
try:
    import cupy as cp
except ImportError:
    !pip install cupy-cuda12x -q
    import cupy as cp

print("CuPy version:", cp.__version__)
print("GPU:", cp.cuda.runtime.getDeviceProperties(0)['name'])

CuPy version: 14.0.1
GPU: b'Tesla T4'


In [19]:
train_u_gpu = cp.asarray(train_u)
train_i_gpu = cp.asarray(train_i)
train_r_gpu = cp.asarray(train_r)

test_u_gpu = cp.asarray(test_u)
test_i_gpu = cp.asarray(test_i)
test_r_gpu = cp.asarray(test_r)

U_embed_gpu = cp.asarray(U_embed)
V_embed_gpu = cp.asarray(V_embed)

global_mean_gpu = float(global_mean)
print("Data moved to GPU")

Data moved to GPU


In [20]:
def epanechnikov_kernel_gpu(dist, bandwidth):
    ratio = dist / bandwidth
    return cp.where(ratio < 1.0, 0.75 * (1.0 - ratio ** 2), 0.0)

In [21]:
def train_local_model_gpu(train_u, train_i, train_r, weights, mask,
                           n_users, n_items, rank=5, n_epochs=15,
                           lr=0.01, reg=0.05, seed=0, batch_size=512):
    rng = cp.random.RandomState(seed)
    P = rng.normal(0, 0.1, (n_users, rank)).astype(cp.float32)
    Q = rng.normal(0, 0.1, (n_items, rank)).astype(cp.float32)

    u_sub, i_sub, r_sub, w_sub = train_u[mask], train_i[mask], train_r[mask], weights[mask]
    n_samples = len(u_sub)
    if n_samples == 0:
        return P, Q

    for epoch in range(n_epochs):
        perm = rng.permutation(n_samples)
        u_sub, i_sub, r_sub, w_sub = u_sub[perm], i_sub[perm], r_sub[perm], w_sub[perm]

        for start in range(0, n_samples, batch_size):
            end = start + batch_size
            bu, bi, br, bw = u_sub[start:end], i_sub[start:end], r_sub[start:end], w_sub[start:end]

            pu, qi = P[bu], Q[bi]
            pred = cp.sum(pu * qi, axis=1)
            err = (br - pred) * bw

            grad_p = -2 * err[:, None] * qi + 2 * reg * pu
            grad_q = -2 * err[:, None] * pu + 2 * reg * qi

            cp.add.at(P, bu, -lr * grad_p)
            cp.add.at(Q, bi, -lr * grad_q)

    return P, Q

In [22]:
def train_llorma_gpu(train_u, train_i, train_r, U_embed, V_embed,
                      n_users, n_items, n_anchors=20, rank=5,
                      bandwidth=0.8, n_epochs=15, lr=0.01, reg=0.05):
    rng = cp.random.RandomState(RANDOM_SEED)
    anchor_positions = rng.choice(len(train_u), size=n_anchors, replace=False)
    local_models = []

    for t in range(n_anchors):
        pos = int(anchor_positions[t])
        au, ai = int(train_u[pos]), int(train_i[pos])

        du = cp.linalg.norm(U_embed - U_embed[au], axis=1)
        di = cp.linalg.norm(V_embed - V_embed[ai], axis=1)

        wu = epanechnikov_kernel_gpu(du, bandwidth)
        wi = epanechnikov_kernel_gpu(di, bandwidth)

        sample_w = wu[train_u] * wi[train_i]
        mask = sample_w > 1e-6
        if int(mask.sum()) < 20:
            continue

        P, Q = train_local_model_gpu(train_u, train_i, train_r, sample_w, mask,
                                      n_users, n_items, rank=rank, n_epochs=n_epochs,
                                      lr=lr, reg=reg, seed=t)

        local_models.append({'au': au, 'ai': ai, 'P': P, 'Q': Q, 'wu': wu, 'wi': wi})
        print(f"Anchor {t+1}/{n_anchors} trained | neighbors used: {int(mask.sum())}")

    return local_models

In [23]:
def predict_llorma_gpu(local_models, users, items, global_mean):
    preds = cp.zeros(len(users), dtype=cp.float64)
    weight_sums = cp.zeros(len(users), dtype=cp.float64)

    for m in local_models:
        w = m['wu'][users] * m['wi'][items]
        active = w > 1e-6
        if not cp.any(active):
            continue
        local_pred = cp.sum(m['P'][users[active]] * m['Q'][items[active]], axis=1)
        preds[active] += w[active] * local_pred
        weight_sums[active] += w[active]

    covered = weight_sums > 1e-6
    preds[covered] /= weight_sums[covered]
    preds[~covered] = global_mean

    return cp.clip(preds, 0.5, 5.0)

In [24]:
# ============================================================
# LLORMA GPU - 5 RUN HARDWARE EXPERIMENT
# ============================================================

N_RUNS = 5

N_ANCHORS = 20
RANK = 5
BANDWIDTH = 0.8
N_EPOCHS = 15
LR = 0.01
REG = 0.05


llorma_gpu_runs = []


# ============================================================
# GPU WARM-UP
# ============================================================

print("CuPy GPU warm-up...")

warmup_array = cp.zeros(
    (1000, 1000),
    dtype=cp.float32
)

warmup_result = cp.sum(
    warmup_array
)

cp.cuda.Stream.null.synchronize()

del warmup_array
del warmup_result

print("Warm-up complete.")


# ============================================================
# FIVE ACTUAL RUNS
# ============================================================

for run in range(1, N_RUNS + 1):

    print(
        f"\n{'=' * 25}"
        f" LLORMA GPU RUN {run}/5 "
        f"{'=' * 25}"
    )

    # --------------------------------------------------------
    # Ensure GPU is idle before starting
    # --------------------------------------------------------

    cp.cuda.Stream.null.synchronize()

    # --------------------------------------------------------
    # Start monitor
    # --------------------------------------------------------

    monitor = HardwareMonitor(
        interval=0.2
    )

    monitor.start()

    start_time = time.perf_counter()

    # --------------------------------------------------------
    # EXACT EXISTING LLORMA GPU TRAINING
    # --------------------------------------------------------

    local_models_gpu = train_llorma_gpu(
        train_u_gpu,
        train_i_gpu,
        train_r_gpu,

        U_embed_gpu,
        V_embed_gpu,

        n_users,
        n_items,

        n_anchors=N_ANCHORS,
        rank=RANK,
        bandwidth=BANDWIDTH,
        n_epochs=N_EPOCHS,
        lr=LR,
        reg=REG
    )

    # --------------------------------------------------------
    # VERY IMPORTANT:
    # CuPy operations are asynchronous.
    # --------------------------------------------------------

    cp.cuda.Stream.null.synchronize()

    elapsed = (
        time.perf_counter()
        - start_time
    )

    monitor.stop()

    result = monitor.get_results()

    result["run"] = run
    result["training_time_s"] = elapsed

    llorma_gpu_runs.append(
        result
    )

    print(
        f"Training time: {elapsed:.4f} s"
    )


# ============================================================
# SAVE
# ============================================================

llorma_gpu_file = os.path.join(
    RESULT_DIR,
    "LLORMA_GPU_hardware_results.csv"
)

llorma_gpu_results_df = (
    save_five_run_results(
        llorma_gpu_runs,
        llorma_gpu_file,
        "LLORMA",
        "GPU"
    )
)

CuPy GPU warm-up...
Warm-up complete.

========================= LLORMA GPU RUN 1/5 =========================
Anchor 1/20 trained | neighbors used: 6063
Anchor 2/20 trained | neighbors used: 49830
Anchor 3/20 trained | neighbors used: 54372
Anchor 4/20 trained | neighbors used: 12452
Anchor 5/20 trained | neighbors used: 11395
Anchor 6/20 trained | neighbors used: 58042
Anchor 7/20 trained | neighbors used: 43897
Anchor 8/20 trained | neighbors used: 54592
Anchor 9/20 trained | neighbors used: 52533
Anchor 10/20 trained | neighbors used: 29290
Anchor 11/20 trained | neighbors used: 57869
Anchor 12/20 trained | neighbors used: 51386
Anchor 13/20 trained | neighbors used: 55361
Anchor 14/20 trained | neighbors used: 17979
Anchor 15/20 trained | neighbors used: 51850
Anchor 16/20 trained | neighbors used: 12289
Anchor 17/20 trained | neighbors used: 48805
Anchor 18/20 trained | neighbors used: 3694
Anchor 19/20 trained | neighbors used: 37417
Anchor 20/20 trained | neighbors used: 56714
T

,avg_cpu_util_percent,peak_cpu_util_percent,avg_ram_mb,peak_ram_mb,avg_gpu_util_percent,peak_gpu_util_percent,avg_gpu_memory_util_percent,peak_gpu_memory_util_percent,avg_vram_mb,peak_vram_mb,avg_gpu_power_w,peak_gpu_power_w,avg_gpu_temperature_c,peak_gpu_temperature_c,gpu_energy_joules,run,training_time_s,Algorithm,Device
0,48.774,53.500,648.913,666.445,12.855,21.000,0.0,0.0,562.188,566.188,26.933,27.245,41.105,42.0,419.923,1,15.783,LLORMA,GPU
1,49.503,53.550,666.579,666.602,17.891,22.000,0.0,0.0,568.624,570.188,27.172,27.457,41.982,42.0,304.567,2,11.360,LLORMA,GPU
2,38.586,53.450,666.608,666.621,11.928,21.000,0.0,0.0,570.188,570.188,27.089,27.359,42.530,43.0,477.933,3,17.850,LLORMA,GPU
3,49.748,53.500,666.626,666.633,18.204,21.000,0.0,0.0,570.299,572.188,27.346,27.457,43.389,44.0,300.547,4,11.116,LLORMA,GPU
4,49.313,53.550,666.637,666.637,17.491,21.000,0.0,0.0,572.188,572.188,27.291,27.457,44.000,44.0,317.109,5,11.659,LLORMA,GPU
5,47.185,53.510,663.073,666.588,15.674,21.200,0.0,0.0,568.697,570.188,27.166,27.395,42.601,43.0,364.016,AVERAGE,13.554,LLORMA,GPU
6,4.820,0.042,7.915,0.081,3.025,0.447,0.0,0.0,3.852,2.449,0.165,0.094,1.141,1.0,80.414,STD,3.073,LLORMA,GPU



Saved result file:
/content/drive/MyDrive/LLORMA_Project/hardware_results/LLORMA_GPU_hardware_results.csv


In [25]:
cp.cuda.Stream.null.synchronize()
t0 = time.time()
test_preds_gpu = predict_llorma_gpu(local_models_gpu, test_u_gpu, test_i_gpu, global_mean_gpu)
cp.cuda.Stream.null.synchronize()
gpu_infer_time = time.time() - t0

test_preds_gpu_cpu = cp.asnumpy(test_preds_gpu)
rmse_gpu = np.sqrt(np.mean((test_preds_gpu_cpu - test_r) ** 2))
mae_gpu  = np.mean(np.abs(test_preds_gpu_cpu - test_r))

print(f"GPU inference time: {gpu_infer_time:.4f}s")
print(f"Test RMSE (GPU): {rmse_gpu:.4f}")
print(f"Test MAE  (GPU): {mae_gpu:.4f}")

GPU inference time: 1.0822s
Test RMSE (GPU): 1.4824
Test MAE  (GPU): 1.1342


## CPU vs GPU Model Performance Comparison

Based on the execution results, here's a comparison of the CPU and GPU implementations of the LLORMA model:

| Metric           | CPU         | GPU         |
| :--------------- | :---------- | :---------- |
| Train time (s)   | 6.37        | 15.71       |
| Infer time (s)   | 0.0984      | 1.1177      |
| RMSE             | 1.4531      | 1.4824      |
| MAE              | 1.0978      | 1.1342      |

**Key Observations and Reasons:**

1.  **Training Time:** The CPU version trained significantly faster (6.37s) than the GPU version (15.71s). This is unexpected, as GPUs are typically much faster for parallelizable tasks like training neural-network-like models. A possible reason for this inverse speedup (GPU is slower) could be:
    *   **Overhead of Data Transfer:** Moving data between the CPU and GPU memory has a significant overhead. If the model is not large enough, or the number of epochs/iterations is not high enough, this overhead can outweigh the benefits of GPU's parallel processing. The current dataset size or model complexity might not be sufficient to fully leverage the GPU's power, leading to this overhead dominating the training time.
    *   **Inefficient CuPy Implementation:** The `train_local_model_gpu` function might not be fully optimized for GPU operations. For instance, frequent conversions between CPU and GPU arrays or operations that don't fully utilize GPU parallelism could lead to slower performance.
    *   **Small Model Size:** The `RANK` and `N_ANCHORS` values (5 and 20 respectively) suggest a relatively small model. GPUs shine with very large matrices and complex computations, where their massive parallelism can be fully exploited. For smaller computations, the sequential nature of CPU processing combined with less data transfer might be more efficient.

2.  **Inference Time:** Similar to training, the CPU version had a much faster inference time (0.0984s) compared to the GPU version (1.1177s). The reasons are likely similar to those for training time: data transfer overhead and potentially less optimized GPU kernel usage for smaller batch sizes or model predictions.

3.  **RMSE and MAE:** Both RMSE and MAE are slightly higher for the GPU model compared to the CPU model. This indicates that the CPU model achieved slightly better prediction accuracy. This minor difference could be due to:
    *   **Numerical Precision:** Differences in floating-point arithmetic between CPU (NumPy) and GPU (CuPy) can lead to slightly different results, especially over many iterations. While both typically use 32-bit or 64-bit floats, the exact implementation details can vary.
    *   **Random Seed Initialization:** Although `RANDOM_SEED` is set for both, the exact random number generation sequence might diverge between NumPy and CuPy, leading to different weight initializations and convergence paths for the local models, ultimately affecting accuracy.

**Conclusion:**

In this specific case, the CPU implementation significantly outperformed the GPU implementation in terms of both training and inference time, and also achieved slightly better accuracy. This suggests that for this particular model configuration and dataset size, the benefits of GPU acceleration were not realized, likely due to overheads associated with GPU utilization for relatively small computational loads or suboptimal GPU code implementation. For larger datasets or more complex models, the GPU would typically be expected to provide substantial speedups.

## LLORMA CPU vs. Enhanced GCMC Model Comparison

Comparing the LLORMA CPU model with the provided Enhanced GCMC model based on RMSE:

| Model             | RMSE   |
| :---------------- | :----- |
| LLORMA CPU        | 1.4531 |
| Enhanced GCMC     | 1.0847 |

**Analysis:**

Based solely on the Root Mean Squared Error (RMSE) metric, the **Enhanced GCMC model appears to be superior in terms of prediction accuracy compared to the LLORMA CPU model.**

*   **RMSE:** A lower RMSE indicates a better fit of the model to the data, meaning the model's predictions are closer to the actual values. The Enhanced GCMC model's RMSE of 1.0847 is significantly lower than LLORMA CPU's RMSE of 1.4531, suggesting that the Enhanced GCMC model makes more accurate predictions on the test set.

*   **Convergence (GCMC context):** The context also mentions that the Enhanced GCMC model achieved faster convergence during training (reaching the Baseline GCMC's final loss by Epoch 16, a reduction of 4 epochs). While we don't have direct training time comparisons with LLORMA, this indicates improved efficiency within the GCMC framework itself.

**Conclusion:**

For the task of rating prediction, based on the provided RMSE values, the **Enhanced GCMC model demonstrates better predictive performance than the LLORMA CPU model.** Further comparison would ideally include training times, inference times, and other relevant metrics (like MAE for LLORMA CPU vs GCMC if available) to provide a more comprehensive evaluation across all aspects of model performance and efficiency.